# 模組 1 · 入門起步
## nirs4all 互動教學 — 你的第一條 pipeline

本模組帶你寫出第一條 NIRS 機器學習 pipeline，並學會回歸、分類與結果視覺化。

**對應官方範例**：`examples/user/01_getting_started/`（U01–U04）

## 🚀 環境設定（請先執行下面這格）

本課程使用 **nirs4all 官方範例資料集（sample_data）**。下面這格會：
1. 從 GitHub 下載 nirs4all（內含 0.9 版函式庫與官方光譜資料集）
2. 安裝函式庫
3. 切換到 `examples/` 目錄（裡面有 `sample_data/`）

> 💡 **為什麼從 GitHub 安裝？** PyPI（`pip install nirs4all`）目前是舊版 0.4，沒有本課程使用的 `nirs4all.run()` 簡易 API。GitHub 上的版本（0.9+）才有，且需要 **Python ≥ 3.11**（Colab 預設 3.12，沒問題）。

第一次執行約需 1–2 分鐘，請耐心等候出現「✅ 完成」。

In [ ]:
# === Colab / Jupyter 環境設定（每本 notebook 開頭執行一次）===
import os, sys, subprocess

if not os.path.isdir('nirs4all'):
    print('⬇️  下載 nirs4all 與官方資料集（約 1–2 分鐘）…')
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/GBeurier/nirs4all.git'], check=True)
    print('📦 安裝 nirs4all …')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    './nirs4all'], check=True)

# 切換到 examples 目錄（內含 sample_data）
if os.path.basename(os.getcwd()) != 'examples' and os.path.isdir('nirs4all/examples'):
    os.chdir('nirs4all/examples')

import nirs4all
print('✅ 完成！nirs4all 版本：', nirs4all.__version__)
print('   工作目錄：', os.getcwd())
print('   可用資料集：', [d for d in os.listdir('sample_data') if os.path.isdir(os.path.join('sample_data', d))])

---
## U01 · Hello World：你的第一條 pipeline  ★☆☆☆☆

**學習目標**：用 `nirs4all.run()` 訓練最小 pipeline、認識 list 結構、讀取 `RunResult`。

nirs4all 的 pipeline 就是一個**步驟清單（list）**。最小流程四步：特徵縮放 → 目標縮放 → 交叉驗證切分 → 模型。執行後得到 `result`，用 `.best_rmse`、`.best_r2`、`.top(n)` 看成績。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression

result = nirs4all.run(
    pipeline=[
        MinMaxScaler(),                            # 特徵縮放到 [0,1]
        {"y_processing": MinMaxScaler()},          # 目標縮放
        ShuffleSplit(n_splits=3, test_size=0.25),  # 3 折交叉驗證
        {"model": PLSRegression(n_components=10)},  # PLS 模型
    ],
    dataset="sample_data/regression",
    name="HelloWorld",
    verbose=1,
)

print(f"\n最佳 RMSE: {result.best_rmse:.4f}")
print(f"最佳 R²:  {result.best_r2:.4f}")
print(f"預測總數: {result.num_predictions}")
for i, p in enumerate(result.top(n=3, display_metrics=['rmse', 'r2']), 1):
    print(f"{i}. {p['model_name']} — RMSE {p['rmse']:.4f}, R² {p['r2']:.4f}")

> ✍️ **動手練習**：把 `n_components` 改成 5 與 20，比較 RMSE。成分太少會欠擬合、太多會過擬合 — 你的資料最佳值是多少？

---
## U02 · 基礎回歸：NIRS 前處理與模型比較  ★★☆☆☆

**學習目標**：套用 NIRS 前處理、用 `feature_augmentation` 自動探索組合、用 `PredictionAnalyzer` 視覺化。

真實光譜需要前處理對抗散射與雜訊。`feature_augmentation` 會自動產生多種前處理組合並各自建模比較，省去手動排列。最後用 `PredictionAnalyzer` 畫圖。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import (
    Detrend, FirstDerivative, Gaussian, SavitzkyGolay, Haar)
from nirs4all.visualization.predictions import PredictionAnalyzer

pipeline = [
    MinMaxScaler(),
    {"y_processing": MinMaxScaler()},
    {"feature_augmentation": {          # 自動展開前處理組合
        "_or_": [Detrend, FirstDerivative, Gaussian, SavitzkyGolay, Haar],
        "pick": 2,                      # 每次挑 2 種串接
        "count": 3,                     # 產生 3 種組合
    }},
    ShuffleSplit(n_splits=3, test_size=0.25),
]
for n in [5, 10, 15]:
    pipeline.append({"name": f"PLS-{n}", "model": PLSRegression(n_components=n)})

result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="BasicRegression", verbose=1)

print("\n=== 前 5 名 ===")
for i, p in enumerate(result.top(n=5, display_metrics=['rmse', 'r2']), 1):
    print(f"{i}. {p['model_name']} | RMSE {p['rmse']:.4f} | 前處理: {p.get('preprocessings','-')}")

# 視覺化
analyzer = PredictionAnalyzer(result.predictions)
analyzer.plot_top_k(k=3, rank_metric='rmse')
analyzer.plot_heatmap(x_var="model_name", y_var="preprocessings",
                      rank_metric="rmse", display_metric="rmse")

> ✍️ **動手練習**：把 `pick` 改 1、`count` 改 5，找出單一前處理中表現最好的那個。

---
## U03 · 基礎分類：Random Forest  ★★☆☆☆

**學習目標**：建立分類 pipeline、使用分類器、用平衡準確率與混淆矩陣評估。

分類流程與回歸幾乎相同，差別在模型換成分類器、指標換成 `balanced_accuracy`（處理類別不平衡）。資料集改用 `sample_data/binary`。

In [ ]:
import nirs4all
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ShuffleSplit
from nirs4all.operators.transforms import (
    FirstDerivative, StandardNormalVariate, Haar, MultiplicativeScatterCorrection)
from nirs4all.visualization.predictions import PredictionAnalyzer

pipeline = [
    {"feature_augmentation": [FirstDerivative, StandardNormalVariate,
                              Haar, MultiplicativeScatterCorrection]},
    StandardScaler(),
    ShuffleSplit(n_splits=3, test_size=0.25),
    {"model": RandomForestClassifier(n_estimators=50, max_depth=8,
              random_state=42), "name": "RandomForest"},
]

result = nirs4all.run(pipeline=pipeline, dataset="sample_data/binary",
                      name="BasicClassification", verbose=1, random_state=42)

preds = result.predictions
print("\n=== 前 5 名（平衡準確率）===")
for i, p in enumerate(preds.top(5, rank_metric='balanced_accuracy',
        display_metrics=['balanced_accuracy', 'balanced_recall']), 1):
    print(f"{i}. {p['model_name']} — bAcc {p['balanced_accuracy']:.4f}")

analyzer = PredictionAnalyzer(preds)
analyzer.plot_confusion_matrix(k=4, rank_metric='balanced_accuracy')

> ✍️ **動手練習**：執行 `!pip install xgboost` 後，加入 `from xgboost import XGBClassifier` 與一個 `{"model": XGBClassifier(...), "name":"XGBoost"}`，比較它與 RandomForest。

---
## U04 · 視覺化全覽：PredictionAnalyzer  ★★☆☆☆

**學習目標**：熟悉五種圖、理解「排序」與「顯示」分離、認識 `score_scope`。

最關鍵觀念：**排序與顯示分離**。`rank_metric`+`rank_partition` 決定「誰最佳」；`display_metric`+`display_partition` 決定「畫面顯示什麼」。例如可「用驗證集 RMSE 排序，但顯示測試集 R²」。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.model_selection import ShuffleSplit
from nirs4all.operators.transforms import (
    StandardNormalVariate, Gaussian, SavitzkyGolay, Haar)
from nirs4all.visualization.predictions import PredictionAnalyzer

pipeline = [
    MinMaxScaler(), {"y_processing": MinMaxScaler()},
    {"feature_augmentation": [StandardNormalVariate, Gaussian, SavitzkyGolay, Haar]},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=5),  "name": "PLS-5"},
    {"model": PLSRegression(n_components=10), "name": "PLS-10"},
    {"model": Ridge(alpha=1.0),              "name": "Ridge"},
    {"model": ElasticNet(alpha=0.1),         "name": "ElasticNet"},
]
result = nirs4all.run(pipeline=pipeline,
                      dataset=['sample_data/regression', 'sample_data/regression_2'],
                      name="VizDemo", verbose=1)

az = PredictionAnalyzer(result.predictions)
az.plot_top_k(k=3, rank_metric='rmse')
# 用驗證集 RMSE 排序，但顯示測試集 R²
az.plot_heatmap(x_var="model_name", y_var="preprocessings",
                rank_metric='rmse', rank_partition='val',
                display_metric='r2', display_partition='test', aggregation='best')
az.plot_candlestick(variable="model_name", display_metric='rmse')
az.plot_histogram(display_metric='r2')

> ✍️ **動手練習**：把 heatmap 的 `y_var` 改成 `"dataset_name"`，找出哪個模型跨兩個資料集最穩定。